# Thinking Spatially: Mapping Cross-Border Trade  (Solution)

Reference solution for `exercise_5_geospatial_trade_maps.ipynb` — **Lesson 5: GeoPandas**.

**Scenario.** You are a food-security analyst at FEWS NET. You have informal cross-border trade
records for South Africa and you want to put them *on a map*: which neighbours does South Africa
trade with, how much, and along what corridor?

**Data**
- `sa_cross_border_trade.json` — FEWS NET monitored informal trade for South Africa
  ([source](https://fdw.fews.net/api/tradeflowquantityvaluefacts/?dataset=1845&country=ZA&fields=simple&format=json))
- `africa_countries.geojson` — country boundary polygons (Natural Earth)
- `sadc_capitals.csv` — capital-city coordinates for Southern African countries

**Four skills** (one per part): table → GeoDataFrame · spatial join · attribute join + flows ·
multi-layer map.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd

import geopandas as gpd
import contextily as cx
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))
from src.utilities.project_paths import RAW_DIR

GEO_DIR           = RAW_DIR / 'geo'
COUNTRIES_GEOJSON = GEO_DIR / 'africa_countries.geojson'
CAPITALS_CSV      = GEO_DIR / 'sadc_capitals.csv'
TRADE_JSON        = RAW_DIR / 'south_africa' / 'fews_net' / 'sa_cross_border_trade.json'

# Southern-African countries used to zoom the maps (ISO A2 codes)
SADC = ['ZA', 'ZW', 'MW', 'ZM', 'MZ', 'BW', 'NA', 'LS', 'SZ', 'AO', 'TZ']

pd.set_option('display.float_format', '{:,.1f}'.format)
print('geopandas', gpd.__version__, '| contextily', cx.__version__)

---
# Part A — From a Table to a Map

A `GeoDataFrame` is a regular DataFrame plus a `geometry` column that stores shapes (points,
lines, polygons) and knows how to draw them. In this part we build three layers:
the **trade table**, the **country polygons**, and the **capital points**.

## A1. The trade table

Load the FEWS NET records, convert the common unit (kilograms) to tonnes, and total the volume
by trading **partner** (the country that is *not* South Africa).

In [ ]:
raw = pd.DataFrame(json.load(open(TRADE_JSON)))
print(f'{len(raw):,} records | {raw.period_date.min()} to {raw.period_date.max()}')

# tonnes from the common unit (kilograms); partner = the country that is not South Africa
raw['tonnes']      = pd.to_numeric(raw['common_unit_quantity'], errors='coerce') / 1000
raw['partner']     = raw['destination'].where(raw['source'] == 'South Africa', raw['source'])
raw['partner_iso'] = raw['destination_country_code'].where(raw['source'] == 'South Africa',
                                                           raw['source_country_code'])

trade = (raw.groupby(['partner', 'partner_iso'], as_index=False)
            .agg(tonnes=('tonnes', 'sum'), records=('id', 'size'))
            .sort_values('tonnes', ascending=False))
display(trade)

top_products = (raw[raw.source == 'South Africa']
                .groupby('product', as_index=False).agg(tonnes=('tonnes', 'sum'))
                .sort_values('tonnes', ascending=False).head(5))
print('Top commodities leaving South Africa:')
display(top_products)

**Interpretation:** The monitored data is dominated by one corridor — maize and other grains
flowing from **South Africa to Zimbabwe** (~58k tonnes). Malawi appears too, but with no recorded
volume. That sparsity is realistic, and it will matter when we join the trade onto the map.

## A2. The country polygons (GeoJSON)

Load the boundary polygons with `gpd.read_file()`. Notice the `geometry` column holds `POLYGON`
shapes and the layer carries a **CRS** (coordinate reference system).

In [ ]:
countries = gpd.read_file(COUNTRIES_GEOJSON)

print('CRS        :', countries.crs)
print('rows, cols :', countries.shape)
print('geom types :', countries.geometry.geom_type.value_counts().to_dict())
display(countries[['name', 'iso_a2', 'pop_est']].head())

countries.plot(figsize=(7, 7), color='lightgray', edgecolor='white')
plt.title('African country boundaries (GeoJSON)')
plt.show()

**Interpretation:** `EPSG:4326` is plain longitude/latitude in degrees. The GeoJSON stores each
country's border as a `POLYGON` (or `MULTIPOLYGON` for countries with islands) — impossible to
hold in a plain CSV. GeoJSON is human-readable and web-friendly; the same data also ships as
Shapefile (`.shp` + companions) or the single-file GeoPackage (`.gpkg`).

## A3. The capital points

The capitals arrive as plain `lat`/`lon` numbers. Turn them into a points `GeoDataFrame`.

⚠️ `Point` takes `(x, y)` — that is **`(longitude, latitude)`**, not `(lat, lon)`.

In [ ]:
caps = pd.read_csv(CAPITALS_CSV)

capitals = gpd.GeoDataFrame(
    caps,
    geometry=[Point(lon, lat) for lon, lat in zip(caps['lon'], caps['lat'])],
    crs='EPSG:4326',
)
display(capitals.head())

ax = countries[countries.iso_a2.isin(SADC)].plot(figsize=(8, 8), color='lightgray', edgecolor='white')
capitals.plot(ax=ax, color='red', markersize=40)
plt.title('SADC capitals over country boundaries')
plt.show()

---
# Part B — Spatial Join: Which Country Is Each Point In?

A spatial join matches rows by **where they are**, not by a shared ID. Instead of "do these rows
share a key?" we ask "is this point **within** that polygon?" — the *Spatial VLOOKUP*.

## B1. Tag each capital with its country

Use `gpd.sjoin(..., predicate='within', how='left')` to attach the polygon attributes
(`name`, `pop_est`, ...) to every capital point.

In [ ]:
capitals_tagged = gpd.sjoin(capitals, countries, how='left', predicate='within')

print('capitals    :', len(capitals))
print('tagged rows :', len(capitals_tagged))
print('unmatched   :', capitals_tagged['name'].isna().sum())
display(capitals_tagged[['capital', 'country', 'name', 'pop_est']])

**Interpretation:** Every capital fell inside exactly one country polygon, so `name` was
recovered *spatially* — no fragile text-matching on country names. We also got `pop_est` and every
other polygon attribute for free. This is the "Spatial VLOOKUP".

## B2. Why `predicate` and coordinate order matter (Pitfall #1)

What if we had swapped longitude and latitude when building the points? The points land in the
wrong place — often in the ocean — and an **inner** join silently drops them.

In [ ]:
# WRONG on purpose: Point(lat, lon) instead of Point(lon, lat)
swapped = gpd.GeoDataFrame(
    caps,
    geometry=[Point(lat, lon) for lon, lat in zip(caps['lon'], caps['lat'])],
    crs='EPSG:4326',
)

swap_inner = gpd.sjoin(swapped, countries, how='inner', predicate='within')
swap_left  = gpd.sjoin(swapped, countries, how='left',  predicate='within')

print(f'Correct order  -> inner keeps {len(gpd.sjoin(capitals, countries, how="inner", predicate="within"))} of {len(caps)}')
print(f'Swapped order  -> inner keeps {len(swap_inner)} of {len(caps)}  (the rest fell outside every polygon)')
print(f'Swapped order  -> left  keeps {len(swap_left)}  (unmatched become NaN: {swap_left["name"].isna().sum()})')

**Interpretation:** Swapping the coordinates throws the points far from home, so `predicate='within'`
finds no match. With `how='inner'` those rows vanish without warning; with `how='left'` they survive
as `NaN` so you can catch the problem. Two lessons: always use `Point(lon, lat)`, and prefer a left
join so you can *count* the unmatched rows.

---
# Part C — Join the Trade onto the Map, and Build Flows

Now attach the trade volumes to the country polygons (a plain attribute merge on the ISO code),
and build **flow lines** from South Africa to each partner.

## C1. Attribute join: trade tonnes onto polygons

Merge `trade` onto `countries` with a **left** join so every country is kept — partners get a
volume, everyone else gets `NaN` (Pitfall #3: don't drop them with an inner join).

In [ ]:
map_data = countries.merge(
    trade[['partner_iso', 'tonnes']],
    left_on='iso_a2', right_on='partner_iso', how='left',
)

print('countries total          :', len(map_data))
print('countries with trade data:', map_data['tonnes'].notna().sum())
display(map_data.loc[map_data.tonnes.notna(), ['name', 'iso_a2', 'tonnes']])

**Interpretation:** The left join keeps all 51 countries; only South Africa's actual partners
carry a value. Keeping the `NaN` rows is deliberate — on the map they become a neutral "no data"
grey, which is honest. An inner join would have silently shrunk the map to two countries.

## C2. Build flow lines

A flow is a `LineString` from South Africa's capital to a partner's capital. Build one per partner
that has a recorded volume.

In [ ]:
sa_point    = capitals.loc[capitals.iso_a2 == 'ZA', 'geometry'].iloc[0]
partner_pts = capitals.set_index('iso_a2')['geometry']

flow_rows = []
for _, r in trade[trade.tonnes > 0].iterrows():
    if r.partner_iso in partner_pts.index:
        flow_rows.append({'partner': r.partner, 'tonnes': r.tonnes,
                          'geometry': LineString([sa_point, partner_pts[r.partner_iso]])})

flows = gpd.GeoDataFrame(flow_rows, crs='EPSG:4326')
display(flows[['partner', 'tonnes']])

---
# Part D — The Multi-Layer Map

Think of a map as a layer cake: **basemap** at the bottom, **boundaries / choropleth** in the
middle, **your data** (flows, points) on top. Plot from top to bottom and control the stacking with
`zorder`. Convert everything to **Web Mercator (EPSG:3857)** first so `contextily` basemaps line up
(Pitfall #2).

## D1. Assemble the layers

In [ ]:
# Project every layer to Web Mercator for contextily
map_web      = map_data.to_crs(3857)
flows_web    = flows.to_crs(3857)
capitals_web = capitals.to_crs(3857)
sadc_web     = countries[countries.iso_a2.isin(SADC)].to_crs(3857)

fig, ax = plt.subplots(figsize=(11, 11))

# Layer 1: choropleth of trade volume (grey where there is no data)
map_web.plot(ax=ax, column='tonnes', cmap='OrRd', edgecolor='white', linewidth=0.6,
             legend=True, legend_kwds={'label': 'Informal trade with SA (tonnes)', 'shrink': 0.5},
             missing_kwds={'color': '#e8e8e8'}, zorder=2)

# Layer 2: outline South Africa
map_web[map_web.iso_a2 == 'ZA'].plot(ax=ax, facecolor='none', edgecolor='black',
                                     linewidth=2, zorder=3)

# Layer 3: trade flows, line width scaled by tonnage
flows_web['lw'] = 1 + 6 * flows_web['tonnes'] / flows_web['tonnes'].max()
for _, r in flows_web.iterrows():
    ax.plot(*r.geometry.xy, color='navy', linewidth=r['lw'], alpha=0.85, zorder=4)

# Layer 4: capital markers + labels
capitals_web.plot(ax=ax, color='black', markersize=25, zorder=5)
for _, r in capitals_web.iterrows():
    ax.annotate(r['capital'], xy=(r.geometry.x, r.geometry.y),
                xytext=(4, 4), textcoords='offset points', fontsize=8)

# Layer 5: basemap (contextily draws it behind everything)
try:
    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)
except Exception as e:
    print('Basemap skipped (offline?):', type(e).__name__)

# Zoom to the Southern-African region
xmin, ymin, xmax, ymax = sadc_web.total_bounds
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_title("South Africa's informal cross-border trade", fontsize=15, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

**Interpretation:** The map now tells the story at a glance: a thick navy corridor carries grain
from Pretoria to Harare, Zimbabwe glows dark on the choropleth, and the remaining SADC countries
sit in neutral grey ("monitored, no recorded volume"). Basemap for context, boundaries for
reference, data on top — the layer cake.

---
# Part E — Common Pitfalls (Debugging Checklist)

| Symptom | Likely cause | Fix |
|---|---|---|
| Points in the wrong place | lat/lon swapped | use `Point(lon, lat)` |
| Basemap misaligned / tiny dot | wrong projection | `.to_crs(epsg=3857)` before `add_basemap` |
| Rows disappear after `sjoin` | inner join | use `how='left'`, then count `NaN` |
| File won't load | missing shapefile parts | ship the `.zip`/`.gpkg`, not a lone `.shp` |
| Spatial join gives nonsense | CRS mismatch | align with `.to_crs()` before the join |

**When in doubt, plot it** — a quick `.plot()` reveals most problems faster than any print.

### Quick Check ✓

1. What is the difference between a regular `merge()` and a spatial `sjoin()`?
2. Why must you convert to `EPSG:3857` before calling `cx.add_basemap()`?
3. In C1 the left join kept 51 countries but only 2 had trade values. Why keep the other 49?

**Answers**
1. `merge()` matches on a shared key/ID; `sjoin()` matches on geometry — e.g. whether a point is
   `within` a polygon. Same idea, location instead of a key.
2. Contextily tiles are served in Web Mercator (metres). If your data is still in degrees
   (`EPSG:4326`) the basemap and data use different units and won't align.
3. They are real countries with no *recorded* informal trade, not errors. Keeping them (grey "no
   data") is honest and avoids silently shrinking the map — an inner join would have dropped them.